# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshit5445/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook maps my provisional lane from Week 1 onto a concrete ML task. The goal is to define what the system should predict or rank, how success will be measured, what one row represents, and how the output supports a real content decision.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring

**ML task type: Ranking.**

The system will rank pages in order of priority for human refresh review. The practical output is not simply a yes/no label; it is an ordered queue so a reviewer with limited time can inspect the highest-priority pages first.

The ranking can use several page-level signals together, such as impressions, CTR, average position, content type, content characteristics, and trend-related features. This makes the task suitable for ML because the useful priority signal may depend on combinations of several variables rather than one fixed threshold.


In [13]:
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns relevant to the lane:")
print([
    c for c in [
        "content_id", "client_id", "impressions_90d", "ctr",
        "avg_position", "trend_direction", "content_type",
        "word_count", "content_age_days"
    ] if c in df.columns
])


Dataset shape: (30000, 44)
Columns relevant to the lane:
['content_id', 'client_id', 'impressions_90d', 'ctr', 'avg_position', 'trend_direction', 'content_type', 'word_count', 'content_age_days']


## 2. Target or proxy

For the starter version, I will use **`trend_direction` as a proxy outcome** for whether a page is showing a weakening or improving direction.

The proxy is useful for framing and experimentation, but it is not a perfect future target. It describes the observed trend in the starter data rather than proving that a future refresh will succeed.

For a stronger later version of the project, I would define a leakage-safe future outcome using a later observation window, so that the model only uses information available at the time the review decision is made.

For the ranking task, the practical target is therefore:

**Prioritize pages that are more likely to be useful candidates for review, using `trend_direction` as the starter proxy for evaluation.**


In [14]:
# Inspect the proxy outcome used for the starter framing.
print("Trend-direction counts:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nUnique trend-direction values:")
print(sorted(df["trend_direction"].dropna().unique()))


Trend-direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Unique trend-direction values:
['down', 'flat', 'new', 'stable', 'up']


## 3. Success metric

The primary success metric will be **Precision@50**.

Precision@50 asks: **of the first 50 pages placed in the review queue, how many match the defined positive outcome?**

This metric fits the real decision because a reviewer does not have unlimited time. If the system puts useful candidates near the top of the queue, the reviewer can spend the available review time more effectively.

I would also inspect other ranking metrics later, but Precision@50 is the clearest first metric for this starter task because the action is to review a small top-ranked set.

The metric measures the quality of the ranking, not whether a content change itself causes traffic to increase.


In [15]:
# Show the scale of the top-of-queue evaluation.
K = 50
print(f"Primary ranking metric: Precision@{K}")
print(f"At K={K}, each correctly prioritized page contributes {1/K:.3f} to Precision@{K}.")
print("The practical question is how many useful candidates appear in the first 50 recommendations.")


Primary ranking metric: Precision@50
At K=50, each correctly prioritized page contributes 0.020 to Precision@50.
The practical question is how many useful candidates appear in the first 50 recommendations.


## 4. The unit of analysis, as a real dataframe

**Unit of analysis: one content/page record.**

Each row represents one anonymized content item associated with a client. The model should make a ranking decision at this page level because the eventual action  deciding which page to review is also page-level.

The dataframe below shows the fields that describe one page and the proxy outcome used for the starter framing.


In [16]:
# Show the unit of analysis as an actual dataframe.
analysis_columns = [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_type",
    "word_count",
    "content_age_days",
    "trend_direction"
]

available_columns = [c for c in analysis_columns if c in df.columns]

print("One row = one content/page record")
display(df[available_columns].head(10))


One row = one content/page record


,content_id,impressions_90d,ctr,avg_position,content_type,word_count,content_age_days,trend_direction
0,content_304f48230142,3803,0.76,10.6,keyword article,3221.0,187,down
1,content_a1fb4e703a9e,15320,0.05,20.3,keyword article,2481.0,445,down
2,content_9aa793d4d895,12581,0.09,36.5,keyword article,3515.0,141,down
3,content_331d6c4de07b,11751,0.49,6.2,keyword article,NaN,463,stable
4,content_d99b7a2d90ca,19140,0.13,44.0,keyword article,2803.0,263,down
5,content_d4084a4bc775,3970,0.03,8.5,keyword article,3080.0,147,down
6,content_9a34b442b552,20,0.00,7.0,keyword article,3059.0,90,down
7,content_a63219c6e95a,1724,0.06,21.2,keyword article,NaN,445,stable
8,content_5e6c160719bc,32574,0.09,46.0,keyword article,3807.0,90,down
9,content_c27558df2b0c,1240,0.16,4.9,keyword article,NaN,257,down


## 5. Why ML beats a fixed rule here

A fixed rule could say something like:

> “Review every page whose trend is down and whose impressions exceed a chosen threshold.”

That is easy to understand, but it uses manually chosen thresholds and does not naturally capture interactions between many signals.

ML is useful here because page priority may depend on combinations of exposure, CTR, position, content type, age, and other available features. A learned ranking system can estimate a priority score from these signals and produce an ordered queue.

However, ML is not automatically better. It must earn its place through validation against a baseline rule. If a simple rule performs as well as the learned approach, the rule may be preferable because it is easier to explain and maintain.

Therefore, the experiment is:

**fixed baseline rule → learned ranking model → compare on held-out data using Precision@50.**

The model is decision-support for a human reviewer; it does not automatically decide that a page should be changed.


In [17]:
# Inspect a few signals that a learned approach could combine.
feature_columns = [
    "impressions_90d", "ctr", "avg_position",
    "word_count", "content_age_days"
]

present = [c for c in feature_columns if c in df.columns]
print("Candidate numeric signals:", present)

print("\nMissing values in these signals:")
print(df[present].isna().sum())


Candidate numeric signals: ['impressions_90d', 'ctr', 'avg_position', 'word_count', 'content_age_days']

Missing values in these signals:
impressions_90d        0
ctr                    0
avg_position           0
word_count          7699
content_age_days       0
dtype: int64


## 6. Self-check

- [x] My lane is mapped to a concrete ML task type: ranking.
- [x] My target/proxy is clearly identified as `trend_direction`, and its limitations are explained.
- [x] My success metric is Precision@50 and is connected to the real review action.
- [x] My unit of analysis is clearly defined: one row = one content/page record.
- [x] A real dataframe from the starter dataset is shown.
- [x] I explained why ML could be useful compared with a fixed rule.
- [x] I explained that ML must be validated against a baseline rather than assuming it is automatically better.
- [x] The notebook runs from top to bottom without errors.
- [x] The executed notebook is committed to `work/notebooks/w02_ml_task_framing.ipynb`.
- [x] The repository URL is ready to submit.